# Volatility Modeling Demo

This notebook demonstrates the `VolatilityModel` class from the `volatility_forecasting` package.

It shows how to:
1. Load data using `DataLoader`
2. Fit GARCH, GJR-GARCH and EGARCH models
3. Compare models using AIC and BIC
4. Run the Ljung-Box diagnostic test

In [ ]:
import os
import sys

# Works whether Jupyter is launched from project root or notebooks/ folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(project_root, "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from volatility_forecasting.data import DataLoader
from volatility_forecasting.models import VolatilityModel, compare_models

print("Imports OK")

## Step 1: Load Data

We use `DataLoader` to download Netflix (NFLX) stock data and calculate log returns.

In [ ]:
loader = DataLoader(ticker="NFLX")
df = loader.get_processed_data()

print(f"Downloaded {len(df)} rows of data")
df[["date", "close", "log_return"]].tail(5)

## Step 2: Prepare Returns

We scale log returns to percentage points (multiply by 100).
The `arch` library works better with values in this range.

In [ ]:
returns = df["log_return"] * 100
print(f"Returns range: {returns.min():.2f}% to {returns.max():.2f}%")

## Step 3: Fit a GARCH(1,1) Model

In [ ]:
model = VolatilityModel(returns, model_type="GARCH")
model.fit()

print("Model metrics:")
print(model.get_aic_bic())

## Step 4: Ljung-Box Diagnostic Test

Checks if the model captured all patterns in the data.
A p-value above 0.05 means the model is adequate.

In [ ]:
lb_results = model.ljung_box_test(lags=[5, 10, 15])
print("Ljung-Box test results:")
print(lb_results)

## Step 5: Compare All Three Models

We compare GARCH, GJR-GARCH and EGARCH using AIC and BIC.
Lower AIC/BIC means a better model.

In [ ]:
comparison = compare_models(returns)
print("Model comparison (sorted by AIC):")
print(comparison)

## Step 6: Standardized Residuals

We extract standardized residuals from the best model (EGARCH).

In [ ]:
best_model = VolatilityModel(returns, model_type="EGARCH")
best_model.fit()

residuals = best_model.get_standardized_residuals()
print(f"Residuals mean: {residuals.mean():.4f}")
print(f"Residuals std:  {residuals.std():.4f}")
residuals.tail(5)